# Kaggle ComfyUI + LTX 2.3 GGUF v18
Model: Q3_K_M | Full pipeline: T2V, I2V, ICLoRA
---

In [ ]:
MODEL_VARIANT='Q3_K_M'
DOWNLOAD_MODELS=True
LOW_VRAM_MODE=True
print(f'Config: {MODEL_VARIANT}, LowVRAM: {LOW_VRAM_MODE}')

In [ ]:
import os,sys,subprocess,shutil,time
from pathlib import Path
WORKING=Path('/kaggle/working');COMFY=WORKING/'ComfyUI';VENV=WORKING/'venv'
start=time.time();print('Installing...')
if not VENV.exists():
    subprocess.run([sys.executable,'-m','pip','install','-q','virtualenv'],check=True)
    subprocess.run(['virtualenv',str(VENV),'-p','/usr/bin/python3.10'],check=True)
PY=str(VENV/'bin'/'python3.10');PIP=[PY,'-m','pip','install','-q']
subprocess.run([*PIP,'torch','torchvision','torchaudio','--index-url','https://download.pytorch.org/whl/cu124'],check=True)
subprocess.run([*PIP,'requests','einops','opencv-python','Pillow'],check=True)
if not COMFY.exists():
    subprocess.run(['git','clone','--branch','ComfyUI_ltx_2_3_compliant_19_03_2026',
        'https://github.com/Isi-dev/ComfyUI',str(COMFY)],check=True)
    subprocess.run([*PIP,'-r',str(COMFY/'requirements.txt')],check=True)
for n,u in [('ComfyUI-GGUF','https://github.com/city96/ComfyUI-GGUF'),
    ('ComfyUI-LTXVideo','https://github.com/Lightricks/ComfyUI-LTXVideo'),
    ('ComfyUI-Manager','https://github.com/Comfy-Org/ComfyUI-Manager'),
    ('rgthree-comfy','https://github.com/rgthree/rgthree-comfy.git')]:
    p=COMFY/'custom_nodes'/n
    if not p.exists():
        subprocess.run(['git','clone',u,str(p)],capture_output=True,check=False)
        if (p/'requirements.txt').exists():
            subprocess.run([*PIP,'-r',str(p/'requirements.txt')],capture_output=True,check=False)
for d in ['unet','vae','text_encoders','loras','controlnet']:
    dst=Path(f'/tmp/models/{d}');dst.mkdir(parents=True,exist_ok=True)
    src=COMFY/'models'/d;src.mkdir(parents=True,exist_ok=True)
    for f in list(src.iterdir()):
        if f.is_file() or f.is_symlink():f.unlink()
        else:shutil.rmtree(str(f))
    if not src.is_symlink():shutil.rmtree(str(src));src.symlink_to(dst)
print(f'Env done ({time.time()-start:.0f}s)')

In [ ]:
import urllib.request,time,json,subprocess
from pathlib import Path
MDIR=Path('/tmp/models')
subprocess.run(['df','-h','/tmp','/kaggle/working'])

# 1. Main model (10.8 GB)
model_url=f'https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main/ltx-2.3-22b-dev-{MODEL_VARIANT}.gguf'
model_dst=MDIR/'unet'/f'ltx-2.3-22b-dev-{MODEL_VARIANT}.gguf'
t0=time.time()
if DOWNLOAD_MODELS:
    if model_dst.exists() and model_dst.stat().st_size>1e6:
        print(f'SKIP Model: {model_dst.stat().st_size/1e9:.1f} GB')
    else:
        print('DL Model (10.8 GB)...');urllib.request.urlretrieve(model_url,model_dst)
        print(f'  Model: {model_dst.stat().st_size/1e9:.1f} GB')

# 2. VAE (1.1 GB)
vae_url='https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main/vae/ltx-2.3-22b-dev_video_vae.safetensors'
vae_dst=MDIR/'vae'/'ltx-2.3-22b-dev_video_vae.safetensors'
if not vae_dst.exists() or vae_dst.stat().st_size<1e6:
    print('DL VAE (1.1 GB)...');urllib.request.urlretrieve(vae_url,vae_dst)
    print(f'  VAE: {vae_dst.stat().st_size/1e9:.1f} GB')
else: print(f'SKIP VAE: {vae_dst.stat().st_size/1e9:.1f} GB')

# 3. ICLoRA LoRA (180 MB)
iclora_url='https://huggingface.co/Lightricks/LTX-2.3/resolve/main/ltx-2.3-22b-ic-lora-hdr-0.9.safetensors'
iclora_dst=MDIR/'loras'/'ltx-2.3-22b-ic-lora-hdr-0.9.safetensors'
if not iclora_dst.exists() or iclora_dst.stat().st_size<1e6:
    print('DL ICLoRA (180 MB)...');urllib.request.urlretrieve(iclora_url,iclora_dst)
    print(f'  ICLoRA: {iclora_dst.stat().st_size/1e9:.1f} GB')
else: print(f'SKIP ICLoRA: {iclora_dst.stat().st_size/1e9:.1f} GB')

# 4. Distilled LoRA (490 MB)
lora_url='https://huggingface.co/Lightricks/LTX-2.3/resolve/main/ltx-2.3-22b-distilled-lora-384-1.1.safetensors'
lora_dst=MDIR/'loras'/'ltx-2.3-22b-distilled-lora-384-1.1.safetensors'
if not lora_dst.exists() or lora_dst.stat().st_size<1e6:
    print('DL LoRA (490 MB)...');urllib.request.urlretrieve(lora_url,lora_dst)
    print(f'  LoRA: {lora_dst.stat().st_size/1e9:.1f} GB')
else: print(f'SKIP LoRA: {lora_dst.stat().st_size/1e9:.1f} GB')

total=sum(f.stat().st_size for f in MDIR.rglob('*')if f.is_file())/1e9
print(f'\nTotal: {total:.1f} GB ({time.time()-t0:.0f}s)')

In [ ]:
# Download workflows
wd=Path('/kaggle/working/workflows');wd.mkdir(exist_ok=True)
WFS=[('T2V+I2V_Full','https://raw.githubusercontent.com/Lightricks/ComfyUI-LTXVideo/master/example_workflows/2.3/LTX-2.3_T2V_I2V_Single_Stage_Distilled_Full.json'),('ICLoRA_Union','https://raw.githubusercontent.com/Lightricks/ComfyUI-LTXVideo/master/example_workflows/2.3/LTX-2.3_ICLoRA_Union_Control_Distilled.json'),('ICLoRA_HDR','https://raw.githubusercontent.com/Lightricks/ComfyUI-LTXVideo/master/example_workflows/2.3/LTX-2.3_ICLoRA_HDR_Distilled.json'),('Motion_Track','https://raw.githubusercontent.com/Lightricks/ComfyUI-LTXVideo/master/example_workflows/2.3/LTX-2.3_ICLoRA_Motion_Track_Distilled.json'),('V2V_ICLoRA','https://raw.githubusercontent.com/Lightricks/ComfyUI-LTXVideo/master/example_workflows/2.3/LTX-2.3_V2V_ICLoRA_Single_Stage_Distilled.json')]
for n,u in WFS:
    d=wd/f'{n}.json'
    if not d.exists():
     try:urllib.request.urlretrieve(u,d);print(f'  {n}')
     except:print(f'  FAIL {n}')
with open(wd/'workflows.json','w') as f:json.dump(dict(WFS),f)
print('Workflows OK')

In [ ]:
import urllib.request,subprocess,os,time,threading,re
from pathlib import Path
COMFY=Path('/kaggle/working/ComfyUI')
PY=str(Path('/kaggle/working/venv/bin/python3.10'))
subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,index','--format=csv,noheader'])
subprocess.Popen([PY,str(COMFY/'main.py'),'--listen','127.0.0.1','--port','8188','--highvram'],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
print('ComfyUI started')
for i in range(30):
    time.sleep(2)
    try:
        urllib.request.urlopen('http://127.0.0.1:8188/system/stats',timeout=3)
        print(f'API ready ({i*2}s)');break
    except:pass
print('Tunnel...')
URL_FILE='/kaggle/working/url.txt'
def tunnel():
    import re
    p=subprocess.Popen(['ssh','-p','443','-o','StrictHostKeyChecking=no','-R0:localhost:8188','a.pinggy.io'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,universal_newlines=True)
    for line in p.stdout:
        m=re.search(r'(https://[^\s]+\.pinggy\.(?:link|net|io)[^\s]*)',line)
        if m:
            url=m.group(1).strip()
            if 'dashboard' in url:continue
            open(URL_FILE,'w').write(url)
            print(f'TUNNEL URL: {url}')
            break
threading.Thread(target=tunnel,daemon=True).start()
for i in range(60):
    time.sleep(3)
    try:
        u=open(URL_FILE).read().strip()
        if u:print(f'URL: {u}');break
    except:pass
else:print('No URL')